<a href="https://colab.research.google.com/github/invegat-14/RESTAPI/blob/main/Copy_of_minMSE11_cuda_volclose.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# minMSE11-cuda-volclose

This copy of the notebook explicitly uses **Close** (price) as the target series and **Volume** as an additional feature when matching patterns to predict future Close.

In [34]:
SHOW_ENV = False
import requests
from sklearn.metrics import mean_squared_error
import numpy as np
import torch
import pandas as pd
from datetime import datetime, timedelta
import sys
import yfinance as yf
import matplotlib.pyplot as plt
import math
import time
# from polygon import RESTClient
# import heapq
from queue import PriorityQueue
import os
import copy
import cloudpickle
import torch.nn.functional as F
from typing import TypedDict

In [33]:
!unzip /content/sample_data/15.zip

Archive:  /content/sample_data/15.zip
replace AAPL.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [36]:
# Replace with your Polygon.io API key
# API_KEY = 'HNZE4DAvb4z5ux2asvBKz_1WGusD_PIR'
# aws_access_key_id='0f2fcb6c-d38d-4862-b4d7-8955288d5c11'
# aws_secret_access_key='HNZE4DAsvb4z5ux2asvBKz_1WGusD_PIR'
# names = ['APPLE','Microsoft','AMAZON', 'NVDIA', 'TESLA','Google','Meta','Berkshire Hathaway Inc. Class B',
#          'UnitedHealth Group Inc', 'Visa Inc','Exxon', 'JPMorgan Chase & Co', 'Johnson & Johnson',
#          'Walmart Inc', 'Mastercard Incorporated','Procter & Gamble Co','Chevron', 'Home Depot Inc',
#          'Eli Lilly and Company','AbbVie Inc','Bank of America','Merck & Co Inc', 'Broadcom Inc','Coca-Cola Company',
#          'Oracle Corp','PepsiCo Inc','Pfizer Inc', 'Thermo Fisher Scientific Inc', 'Costco Wholesale Corporation',
#          'Invesco QQQ', 'CrowdStrike Holdings', 'Trade Desk Inc']

# SYMBOLS = ["AAPL", "MSFT", "AMZN", "NVDA", "TSLA",
#            "GOOGL",  "META","BRK-B",
#            "UNH", "V", "XOM", "JPM", "JNJ", "WMT", "MA", "PG", "CVX", "HD", "LLY", "ABBV",
#            "BAC", "MRK", "AVGO", "KO", "ORCL", "PEP", "PFE", "TMO", "COST", "QQQ", "CRWD", "TTD"]
# SYMBOLS = ["CVX", "HD", "LLY", "ABBV",
#            "BAC", "MRK", "AVGO", "KO", "ORCL", "PEP", "PFE", "TMO", "COST", "QQQ", "CRWD", "TTD"]
#SYMBOLS = ["AAPL","BAC", "MRK", "AVGO",'LIN', 'UNH']
SYMBOLS = ['NVDA', 'MSFT', 'AAPL', 'GOOG', 'GOOGL']
interval = "1d"
days_of_data = "5y"

In [37]:
# client = RESTClient(api_key=API_KEY, pagination=False)
cloudpickle.pickle.HIGHEST_PROTOCOL

5

In [38]:
SEQ_LEN = 60  # 60 minutes
BIZ_DAYS = 7
INTERVALS_PER_DAY = 1 # 390  # 1-min trading intervals
MSE = True
#DEVICE = 'xpu' if torch.xpu.is_available() else 'cpu';
DEVICE='cpu'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE)

cpu


In [39]:
# def get_company_name(symbol):
#     stock = yf.Ticker(symbol)
#     return stock.info.get('longName', 'Company name not found')

In [40]:
# def get_company_name(symbol):
#     url = f"https://financialmodelingprep.com/api/v3/profile/{symbol}?apikey=JQaWGMuxbuiBJhYvhsiw17NISCaGMXWw"
#     response = requests.get(url)
#     if response.ok:
#         data = response.json()
#         if data:
#             return data[0].get("companyName")
#     return None

In [41]:
def fetch_polygon_data(symbol, days):
    print(f'symbol {symbol}  days {days}')
    # trades = [t for t in client.list_trades(ticker=symbol, limit 1024]
    # df = yf.download(symbol, period='5y',auto_adjust=True)
    # df = df.reset_index()
    # df.to_csv(f"./yahoo15/{symbol}.csv", index=False)
    df = pd.read_csv(f"/content/{symbol}.csv", skiprows=2,header=0,
                     names=['Date','Close','High','Low','Open','Volume'])

    # end = datetime.now(UTC)
    # start = end{symbol- timedelta(days=days_of_data)
    # # url = f"https://api.polygon.io/v2/aggs/ticker/{symbol}/range/{days}/day/{start.date()}?adjusted=true&sort=asc&limit=50000&apiKey={API_KEY}"
    # # url = f'https://api.polygon.io/v2/aggs/ticker/{symbol}/range/{days}/day/{start.date()}/{end.date()}?adjusted=true&apiKey={API_KEY}
    # url = f"https://api.polygon.io/v2/aggs/ticker/{symbol}/range/1/minute/{start.date()}/{end.date()}?adjusted=true&sort=asc&limit=50000&apiKey={API_KEY}"
    # response = requests.get(url)
    # data = response.json()
    # df = pd.DataFrame(data["results"])

    df["Date"] = pd.to_datetime(df["Date"])
    df.set_index("Date", inplace=True)
    # # df.rename(columns={"o": "Open", "h": "High", "l": "Low", "c": "Close", "v": "Volume"}, inplace=True)
    new_column_order = ['Close','Open','High', 'Low','Volume']
    df = df[new_column_order]
    # # time.sleep(13)
    return df

In [42]:
tdf=fetch_polygon_data('AAPL',22)
print(len(tdf))
print(tdf.head())
tdf.tail()
print(tdf.index[-1])
dfDate = pd.to_datetime(tdf.index[-1])
print(dfDate)

symbol AAPL  days 22
1254
                 Close        Open        High         Low     Volume
Date                                                                 
2020-12-07  120.464714  119.062940  121.262945  119.004536   86712000
2020-12-08  121.077995  121.068266  121.662073  119.822241   82225500
2020-12-09  118.547028  121.224022  122.606322  117.787736  115089200
2020-12-10  119.968277  117.301019  120.581557  116.960312   81312200
2020-12-11  119.160309  119.179775  119.501016  117.349687   86939800
2025-12-03 00:00:00
2025-12-03 00:00:00


In [43]:
stockData = []
SYMBOLS = []

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# SYMBOLS = ["AAPL","BAC", "MRK", "AVGO",'LIN', 'UNH']
# for symbol in SYMBOLS:
#         # fn = f"./yahoo2/{symbol}.csv"
#         # if os.path.exists(fn):
#         stockData.append(fetch_polygon_data(symbol,4096))


In [44]:
ssnDf = pd.read_csv('/content/sample_data/top_200_by_market_cap.csv')

In [45]:
ssnDf.head()

,Symbol,Name,MarketCap
0,NVDA,NVIDIA Corporation,4236614041600
1,MSFT,Microsoft Corporation,3895798530048
2,AAPL,Apple Inc.,3003400323072
3,GOOG,Alphabet Inc.,2291784744960
4,GOOGL,Alphabet Inc.,2291782909952


In [46]:
for i, row in enumerate(ssnDf.itertuples(index=False, name='ssn')):
    # fn = f"./yahoo2/{row.Symbol}.csv"
    # if os.path.exists(fn):
    stockData.append(fetch_polygon_data(row.Symbol,i))
    SYMBOLS.append(row.Symbol)
    # time.sleep(1)


symbol NVDA  days 0
symbol MSFT  days 1
symbol AAPL  days 2
symbol GOOG  days 3
symbol GOOGL  days 4
symbol AMZN  days 5
symbol META  days 6
symbol AVGO  days 7
symbol TSLA  days 8
symbol JPM  days 9
symbol WMT  days 10
symbol ORCL  days 11
symbol LLY  days 12
symbol V  days 13
symbol MA  days 14
symbol NFLX  days 15
symbol XOM  days 16
symbol COST  days 17
symbol JNJ  days 18
symbol HD  days 19
symbol PLTR  days 20
symbol PG  days 21
symbol ABBV  days 22
symbol BAC  days 23
symbol CVX  days 24
symbol KO  days 25
symbol GE  days 26
symbol AMD  days 27
symbol TMUS  days 28
symbol CSCO  days 29
symbol PM  days 30
symbol WFC  days 31
symbol CRM  days 32
symbol IBM  days 33
symbol MS  days 34
symbol ABT  days 35
symbol GS  days 36
symbol MCD  days 37
symbol INTU  days 38
symbol LIN  days 39
symbol UNH  days 40
symbol RTX  days 41
symbol BX  days 42
symbol DIS  days 43
symbol AXP  days 44
symbol CAT  days 45
symbol MRK  days 46
symbol T  days 47
symbol PEP  days 48
symbol NOW  days 49
symbo

In [47]:
def get_company_name(symbol):
    return ssnDf[ssnDf['Symbol'] == symbol].iloc[0].Name

In [48]:
print(get_company_name('UNH'))

UnitedHealth Group Incorporated


In [49]:
df = fetch_polygon_data('AAPL',30)

symbol AAPL  days 30


In [50]:
a = df['Close'].tolist()

In [ ]:
target=a[-INTERVALS_PER_DAY*BIZ_DAYS:]

In [ ]:
len(target)

7

In [ ]:
source = a[:-INTERVALS_PER_DAY*BIZ_DAYS]

In [ ]:
len(source)

1247

In [51]:
def verify_lengths():
    for i, symbol in enumerate(SYMBOLS):
        df = stockData[i]
        # print(f"symbol {symbol} has lenght {len(df)}")
        if len(df) < 100:
            raise ValueError(f" {symbol} is too short {len(df)}")

In [52]:
verify_lengths()

In [53]:
print(get_company_name('AAPL'))

Apple Inc.


In [54]:
@torch.no_grad()
def match_pattern_top_k(df, exluded,  x, feature_cols, weights, k=5):
    A = df[feature_cols].values        # Data matrix (N × D)
    N, D = A.shape
    W = np.array(weights).reshape(1, 1, D)

    # Query = last x rows (shape x×D)
    query = A[-x:]

    # Build sliding windows: shape (N-x+1, x, D)
    windows = np.lib.stride_tricks.sliding_window_view(A[0:-(exluded+INTERVALS_PER_DAY*BIZ_DAYS )], (x, D))[:, 0, :, :]

    # Weighted Manhattan distance
    diff = np.abs(windows - query) * W
    dist = diff.sum(axis=(1, 2))  # shape (N-x+1,)

    # Compute MSE for each window (unweighted)
    mse_all = ((windows - query) ** 2).mean(axis=(1, 2))  # mean of all values in window

    # Sort candidate windows by similarity score
    sorted_idx = np.argsort(dist)

    selected = []
    used_ranges = []

    for idx in sorted_idx:
        if len(selected) == k:
            break

        start = idx
        end = idx + x - 1

        # Ensure non-overlapping windows
        overlap = False
        for s, e in used_ranges:
            if not (end < s or start > e):  # ranges overlap
                overlap = True
                break

        if overlap:
            continue

        # Accept this match
        selected.append((idx, dist[idx], mse_all[idx]))
        used_ranges.append((start, end))

    # Return results
    # results = [
    #     {

    #         "mi": idx,
    #         "mse": mse_val
    #         # "distance": dist_val,
    #         # "block": df.iloc[idx:idx + x]
    #     }

    #     for idx, dist_val, mse_val in selected
    # ]
    mis = [s[0] for s in selected]
    mses = [s[2] for s in selected]


    return (mis,mses)


In [ ]:


# @torch.no_grad()
def n_best_mle_multifeature(
    data: torch.Tensor,
    target: torch.Tensor,
    n: int = 5,
    non_overlapping: bool = True,
    device: str = DEVICE,
):
    """
    Find up to `n` best maximum-likelihood matches of a multi-feature target in
    multi-feature data using CUDA (via conv1d operations).

    Parameters
    ----------
    data : torch.Tensor, shape (L, C)
        Long signal (L samples) with C features/channels.
    target : torch.Tensor, shape (W, C)
        Pattern to match (W samples) with same C channels.
    n : int
        Number of best matches to return.
    non_overlapping : bool
        If True, suppress overlapping windows of width W (in start-index space).
    device : str or None
        'cuda' or 'cpu'. If None, uses 'cuda' if available.

    Returns
    -------
    indices : torch.LongTensor, shape (k,)
        Start indices (0-based) of selected windows in data (k <= n).
    scores : torch.FloatTensor, shape (k,)
        ZNCC scores in [-1, 1], higher is better.
    """
    # if data.dim() != 2 or target.dim() != 2:
    #     raise ValueError("data must be (L, C) and target must be (W, C)")

    # L, C = data.shape
    # W, Ct = target.shape
    # if C != Ct:
    #     raise ValueError(f"Channel mismatch: data has {C}, target has {Ct}   L {L}  W {W}")

    # if L < W:
    #     return torch.empty(0, dtype=torch.long), torch.empty(0)

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    x = data.to(device=device, dtype=torch.float32)
    t = target.to(device=device, dtype=torch.float32)

    # zero-mean per-channel target
    t = t - t.mean(dim=0, keepdim=True)  # shape (W, C)
    # energy of flattened target vector
    t_energy = torch.clamp((t * t).sum(), min=1e-12)

    # prepare input for conv1d: (N=1, C, L)
    x4 = x.t().unsqueeze(0)  # (1, C, L)

    # 1) per-channel local sums and sumsq (via conv grouped by channel)
    # ones_kernel = torch.ones(C, 1, W, device=device, dtype=torch.float32)  # (C,1,W)
    # sum_x_per_ch = F.conv1d(x4, ones_kernel, stride=1, groups=C)  # (1, C, L-W+1)
    # sum_x2_per_ch = F.conv1d(x4 * x4, ones_kernel, stride=1, groups=C)  # (1,C,L-W+1)

    # mean_x_per_ch = sum_x_per_ch / W  # (1, C, L-W+1)
    # var_x_per_ch = torch.clamp(sum_x2_per_ch / W - mean_x_per_ch * mean_x_per_ch, min=1e-12)  # (1,C,L-W+1)

    # # flattened-window std: sqrt( sum_c (var_c * W) )
    # sum_var_times_W = (var_x_per_ch * W).sum(dim=1, keepdim=True).squeeze(1)  # (1, L-W+1) -> squeeze -> (L-W+1)
    # std_flat = torch.sqrt(sum_var_times_W).squeeze(0)  # (L-W+1,)

    # 2) cross-correlation: weight shape (out_ch=1, in_ch=C, kernel=W)
    # Build weight by flipping target in time, and placing per-channel kernels
    # target t shape (W, C) -> transpose to (C, W), flip last dim
    weight = t.t().flip(dims=[1]).unsqueeze(0)  # (1, C, W)
    dot_xt = F.conv1d(x4, weight, stride=1)  # (1, 1, L-W+1)
    dot_xt = dot_xt.view(-1)  # (L-W+1,)

    # normalized cross-correlation (ZNCC) over flattened vectors
    denom = std_flat * torch.sqrt(t_energy)  # (L-W+1,)
    # avoid divide-by-zero
    denom = torch.clamp(denom, min=1e-12)
    zncc = dot_xt / denom  # (L-W+1,)

    # clamp to [-1,1] numerically
    zncc = torch.clamp(zncc, -1.0, 1.0)

    # pick top-k with optional non-overlap suppression
    K = min(n, zncc.numel())
    if K <= 0:
        return torch.empty(0, dtype=torch.long), torch.empty(0)

    scores_work = zncc.clone()
    chosen_idx = []
    chosen_scores = []
    for _ in range(K):
        best_score, best_pos = torch.max(scores_work, dim=0)
        if not torch.isfinite(best_score) or best_score <= -1e30:
            break
        chosen_idx.append(best_pos.item())
        chosen_scores.append(best_score.item())
        # if non_overlapping:
        #     left = max(0, best_pos - W + 1)
        #     right = min(zncc.numel() - 1, best_pos + W - 1)
        #     scores_work[left : right + 1] = -1e30
        # else:
        #     scores_work[best_pos] = -1e30

    # indices = torch.tensor(chosen_idx, dtype=torch.long, device="cpu")
    # scores = torch.tensor(chosen_scores, dtype=torch.float32, device="cpu")
    # return (indices, scores)
    return (chosen_idx, chosen_scores)

In [ ]:
def getMi_CUDA_MLE(source,target, excluded, k):
    seq_len = len(source) - (excluded+INTERVALS_PER_DAY*BIZ_DAYS)
    n_features = 2
    print(f'seq_len {seq_len}')
    source = np.array(copy.deepcopy(source[0:seq_len]))
    source = torch.from_numpy(source)

    window = len(target)
    target = np.array(copy.deepcopy(target))
    target = torch.from_numpy(target)
    if source.shape[0] < source.shape[1]:
    # If shape is (features, time), fix it
        source = source.T
        target = target.T

    return n_best_mle_multifeature(source,target)


In [55]:
def show(y_test_true, pred_test_true, symbol, mse):
    # print(f'len(y_test_true) {len(y_test_true)}')
    # print(f'len(pred_test_true) {len(pred_test_true)}')
    plt.figure(figsize=(14, 6))
    plt.plot(pred_test_true, label=f'Predicted Test MSE {mse:.6f}')
    plt.plot(y_test_true, label=f'actual Test (7 days) {symbol}')
    plt.legend();
    plt.title(f'Test Set Fit {symbol}  {get_company_name(symbol)}')
    plt.show()
    # plt.figure(figsize=(14, 6))
    # plt.plot(future_true, label=f'7-Day Future Forecast {symbol}')
    # plt.legend();
    # plt.title(f'Future 7-Day Forecast {symbol}')
    # plt.show()

In [56]:
BL=1
BH=2
# BIG = 0.017
BIG = 0.05 #money 146211.28
BIG = 0.075 #money 176077.11
# BIG = 0.1
# BIG=0.1 #money 176077.11
# BIG=0.15 #money  $176,077.11
          #money   $176,405.07

# mia = []
le = 100
op = True
di = True
us = False
pf = True
webull = True
wm = 1.4
# miiMap = [15,10,9,8,7]
miiMap = ([10,10,10,10,10] if MSE else [11,10,9,8,7])
miiMap = [120,110,90,60,10]
miiap = [12,11,9,6,1]
# miiMap=  [130,110,90,70,50]
# miiMap = [170,150,120,80,20]

mhMap = ([0.9,0.9,0.9,0.9,0.9,0.9] if MSE else [0.9,0.8,0.7,0.6,0.5,0.4])
mhMap = [0,0.9,0.8,0.7,0.6,0.5,0.4]

# mhMap=[0,8,4,0.9,0.5,0.3]
# mhMap = [0,1,0.8,0.6,0.7,0.2,0.1]
mm = not MSE
mm =  True
if di:
    w = INTERVALS_PER_DAY*BIZ_DAYS * 5
else:
    w = (10 if pf else 50)  # 128
bl = []
# mis=[]
nmi = 5
ins = True
dbc = False
dvp = False

ls = 'NVDA'
# lsa = ['COIN','BLK', 'AXON', 'CRM', 'DELL', 'BKNG', 'UNH', 'TMO']
# lsa = ['FTNT', 'BLK', 'PLTR', 'AXON', 'VRTX', 'KLAC', 'LLY', 'DASH', 'BKNG', 'TDG', 'AZO','DE', 'EQIX', 'URI', 'ROP',  'PH', 'AFL', 'NVDA', 'MSFT', 'ISRG']
# lsa = ['AAPL', 'BAC', 'MRK']
lsa = ['TDG', 'BKNG', 'RDP', 'BLK', 'DLR', 'EQIX', 'AZO', 'T']
lsa = []
lsa = ['URI', 'ADSK', 'DLR', 'LHX', 'FDX', 'HLT', 'FAST', 'EMR', 'PAYX', 'KLAC', 'ICE', 'TDG', 'BLK', 'MSI', 'PH', 'DUK', 'PCAR', 'ABNB', 'MMM', 'TEL']
lsa=['EQIX', 'BKNG', 'AZO', 'BLK', 'ISRG', 'TDG', 'MSI', 'FTNT', 'NVDA']


In [57]:
class b_dict_mse(TypedDict):
    delta: float
    wprice: float
    adelta: float
    symbol: str
    mse: float
    flags: int
class b_dict(TypedDict):
    delta: float
    wprice: float
    adelta: float
    symbol: str
    flags: int


In [58]:
verify_lengths()
money=100000.00
#     23100.62
#      79708.68
#      99964.35
# dictionary keys
# delta = 'delta_key'
# b='b_key'
# count = 'count_key'
# adelta = 'adelta_key'
# wprice='wprice_key'
#Flag bits
bls =[[] for _ in range(nmi)]
# for mii in range(nmi):
#     for s in range(5):
#         bls[mii].append(s)
#     print(len(bls[mii]))
# for j in range(nmi):
#     mis.append(-1)
amse = [0 for _ in range(nmi)]
t = INTERVALS_PER_DAY*BIZ_DAYS-(INTERVALS_PER_DAY if dfDate.weekday() == 3 else 0) # Thursday is 3
# t = INTERVALS_PER_DAY*2
targetDate = dfDate + timedelta(days=t + (2 if t > (1 if dfDate.weekday() == 3 else 2) else 0), hours=16)   # add the weekend
if ins:
    print(f"t {t}  df date {dfDate} target date {targetDate}  pf {pf}")
for i, symbol in enumerate(SYMBOLS):
    df = stockData[i]
    if ins:
        print(f'Symbol {symbol}  i {i}')
    # print(f'len(df) {len(df)}')
    df['LVolume'] = np.log10(df['Volume'])
    a = df['Close'].tolist()
    v = df['Volume'].tolist()
    cl = df['Close'].tolist()
    if len(a) < 100:
        raise ValueError(f"symbol {symbol} is too short: {len(a)}")
    # t = a[-10:-5]
    nis = []
    flags = 0
    if pf:
        features = ['Close', 'LVolume', 'Open', 'High', 'Low']
        weights = [1.0, 0.9, 0.6, 0.6,0.6]
        (mis,mses) = match_pattern_top_k(df, w, x=10, feature_cols=features, weights=weights,k=5)
    else:
        (mis,mses) = getMi_CUDA_MLE(a,v, a[-INTERVALS_PER_DAY*BIZ_DAYS*2:-INTERVALS_PER_DAY*BIZ_DAYS],
                         v[-INTERVALS_PER_DAY*BIZ_DAYS*2:-INTERVALS_PER_DAY*BIZ_DAYS], w, nmi)


    for mii in range(nmi):    # df = fetch_polygon_data(symbol,30)
        mi,mse = mis[mii], mses[mii]
    # print(f'len(t) {len(t)} a len {len(a[-INTERVALS_PER_DAY*BIZ_DAYS*2:-INTERVALS_PER_DAY*BIZ_DAYS])}')
        # if pf:
        #     (mi,mse) = getMi(a,v, a[-t:],
        #                      v[-t:], w, nis, True)
        # else:
        #     (mi,mse) = getMi(a,v, a[-INTERVALS_PER_DAY*BIZ_DAYS*2:-INTERVALS_PER_DAY*BIZ_DAYS],
        #                      v[-INTERVALS_PER_DAY*BIZ_DAYS*2:-INTERVALS_PER_DAY*BIZ_DAYS], w, nis, False)

        if di:
            pmi = mi + t
        mse = mse/((a[pmi] if di else a[mi])*(wm if webull else 1))
        amse[mii] += mse
        # print(f'pmi {pmi}  mi {mi}')

        # if MSE:
        #     mia.append((mi,mse))
        # else:
        #     mia.append(mi)
       # (mi,mse) = mia[i]
        # b = math.floor(money/8)
        days = t
        # days=6
        if di:
            assert pmi+days < len(a), "mi should be farther away"
                # print(f" pmi {pmi} mi {mi}  days {days}  len a {len(a)}")
            delta = a[pmi+days] - a[pmi]
        else:
            delta = a[mi+INTERVALS_PER_DAY*BIZ_DAYS-INTERVALS_PER_DAY] - a[mi]
        # if ins and symbol in lsa:
        #     print(f'abs(delta)   {abs(delta)}')
        # if abs(delta) > (0.01 if le < 100 else 0.005):
        if True:
            if di:
                aDelta = a[-1] - a[-days]
            else:
                aDelta = a[-1] - a[mi+INTERVALS_PER_DAY*BIZ_DAYS-INTERVALS_PER_DAY]
            if MSE:
                xxt: b_dict_mse = {'delta': delta,'wprice': (a[-1] if pf else
                    (a[-(INTERVALS_PER_DAY*BIZ_DAYS-INTERVALS_PER_DAY)] if di else a[mi]))*(wm if webull else 1),
                   'aDelta': aDelta, 'symbol': symbol, 'mse': mse, 'mii': mii, 'flags': flags }
            else:
                xxt: b_dict = {'delta': delta,'wprice': (a[-1] if pf else
                    (a[-(INTERVALS_PER_DAY*BIZ_DAYS-INTERVALS_PER_DAY)] if di else a[mi]))*(wm if webull else 1),
                   'aDelta': aDelta, 'symbol' : symbol, 'flags': flags}
            if xxt['delta'] / xxt['wprice'] > BIG:
                xxt['flags'] |= BH
            if xxt['delta'] < 0 and xxt['delta'] / xxt['wprice'] <  -BIG:
                xxt['flags'] |= BL
            flags = xxt['flags']



            if ins and symbol in lsa:
                print(f'mii {mii} xxt {xxt}')

            bls[mii].append(xxt)
        # MSE=mean_squared_error(a[-INTERVALS_PER_DAY*BIZ_DAYS:], a[mi:mi+INTERVALS_PER_DAY*BIZ_DAYS])
        # # show(a[mi+INTERVALS_PER_DAY*BIZ_DAYS:mi+INTERVALS_PER_DAY*BIZ_DAYS*2], a[mi:mi+INTERVALS_PER_DAY*BIZ_DAYS],symbol, MSE)
        # if mi >= 0.5 * w:
        #     b = a[int(-0.5*w + mi): int(mi + INTERVALS_PER_DAY * BIZ_DAYS + 0.5*w)]
        # else:
        #     b = a[mi:mi + INTERVALS_PER_DAY * BIZ_DAYS + w]

        # show(a[-w - INTERVALS_PER_DAY * BIZ_DAYS:], b, symbol, MSE)

        if mi + w >= len(a):
            mix = len(a) - (mi + w)
            print('!!!!!!!!!!!!error mix non-zero!!!!!!!!!!')
        else:
            mix = 0
for mii in range(nmi):
    # amsi[mii[ = copy.deepcopy(amsi[mii])
    amse[mii] /= len(SYMBOLS)
    print(f"amse[{mii}] {amse[mii]}")
    print(f"len(bls[mii] {len(bls[mii])}")

first_run_parser = False


t 7  df date 2025-12-03 00:00:00 target date 2025-12-12 16:00:00  pf True
Symbol NVDA  i 0
mii 0 xxt {'delta': -8.1495361328125, 'wprice': 251.42599487304685, 'aDelta': -2.9600067138671875, 'symbol': 'NVDA', 'mse': np.float64(0.0689311944756298), 'mii': 0, 'flags': 0}
mii 1 xxt {'delta': 1.029937744140625, 'wprice': 251.42599487304685, 'aDelta': -2.9600067138671875, 'symbol': 'NVDA', 'mse': np.float64(0.10056271058794161), 'mii': 1, 'flags': 0}
mii 2 xxt {'delta': 3.6300048828125, 'wprice': 251.42599487304685, 'aDelta': -2.9600067138671875, 'symbol': 'NVDA', 'mse': np.float64(0.1067331917178543), 'mii': 2, 'flags': 0}
mii 3 xxt {'delta': 2.9398345947265625, 'wprice': 251.42599487304685, 'aDelta': -2.9600067138671875, 'symbol': 'NVDA', 'mse': np.float64(0.4434789643992462), 'mii': 3, 'flags': 0}
mii 4 xxt {'delta': 9.529464721679688, 'wprice': 251.42599487304685, 'aDelta': -2.9600067138671875, 'symbol': 'NVDA', 'mse': np.float64(1.7463194572323515), 'mii': 4, 'flags': 0}
Symbol MSFT  i 

In [ ]:
first_run_parser = False
with open("bls.pf.pickle", "wb") as outfile:
    cloudpickle.dump(bls,outfile,5)
with open("amse.pf.pickle", "wb") as outfile:
    cloudpickle.dump(amse,outfile,5)

In [ ]:
first_run_parser = False
with open("bls.test.pickle", "wb") as outfile:
    cloudpickle.dump(bls,outfile,5)
with open("amse.test.pickle", "wb") as outfile:
    cloudpickle.dump(amse,outfile,5)

In [ ]:
first_run_parser = False
with open("bls.pf.pickle", "rb") as infile:
    bls = cloudpickle.load(infile)
with open("amse.pf.pickle", "rb") as infile:
    amse = cloudpickle.load(infile)

In [ ]:
first_run_parser = False
with open("bls.test.pickle", "rb") as infile:
    bls = cloudpickle.load(infile)
with open("amse.test.pickle", "rb") as infile:
    amse = cloudpickle.load(infile)

In [59]:
class b_counts(TypedDict):
    count: int
    buy: bool
    b: (b_dict_mse if MSE else b_dict)
    buy_count: int
    put_count: int


In [60]:
# for mii in range(nmi):
#     bls[mii] = [b for b in bls[mii] if abs(b[0])*le-2*b[1] > 0]
money=200000
# BIG = 0.1
# pf=False
sl = []
fs = []
for _ in range(len(SYMBOLS)):
    bc: b_counts = {'count': 0,'buy': True,'b':None, 'buy_count':0, 'put_count': 0}
    sl.append(bc)
    fs.append(0)
for mii in range(nmi):
    bq = bls[mii]
    for b in bq:
        # if ins:
        #     print(f"b in bq {b}")
        index = SYMBOLS.index(b['symbol'])
        buy = (np.sign(b['delta']) > 0)
        if sl[index]['count'] > 0:
            l = sl[index]
            bl = l['b']
            bl['flags'] |= b['flags']
                # if ins:
                #     print(f'adding b[0] {b[0]}  b {b}   bl {bl}')
            bl['delta'] += (b['delta'] * miiMap[mii] if mm else b['delta'])*(amse[0]/(b['mse']*b['wprice']) if MSE else 1)
            #              (b['delta'] * miiMap[mii] if mm else b['delta'])*(amse[mii]/(b['mse']*b['wprice']) if MSE else 1)
            l['buy'] = (bl['delta'] > 0)
            assert l['count'] == mii, "each mii adds one to the couny"
            l['count'] += 1
            if buy:
                l['buy_count'] += 1
            else:
                l['put_count'] += 1
            # if ins and b['symbol'] in lsa:
            #     print(f"after add b {b} {sl[index]}  mii {mii}")
        else:
            l = sl[index]
            l['count'] = 1
            l['buy'] = buy
            # if not first_run_parser:
            l['b'] = copy.deepcopy(b)
            fs[index] = b['delta']
            l['b']['delta'] = (b['delta'] * miiMap[mii] if mm else b['delta'])*(amse[0]/(b['mse']*b['wprice']) if MSE else 1)
            # sl[index] = l
            l['buy_count'] = l['put_count'] = 0
            if buy:
                l['buy_count'] = 1
            else:
                l['put_count'] = 1
            # if ins and b['symbol'] in lsa:
            #     print(f'after first {sl[index]}  mii {mii}')
pfb = []
# gzc = 0
# for index_, t in enumerate(sl):
#     if t['count'] > 0:
#         gzc += 1
# print(f"gzc {gzc}")
for index_, t in enumerate(sl):
    if t['count'] > 0:
        bl = t['b']
        count = abs(t['buy_count'] - t['put_count'])
        # print(f"before assert count {count} t {t}")
        assert count in [1,3,5], "count must be odd 1,3, or 5"
        if ins and bl['symbol'] in lsa:
            print(f"bl {bl} before")
        bl['delta'] /= mhMap[count]
        if ins and bl['symbol'] in lsa:
            print(f"bl {bl} after")
        # if fb[index] != (np.sign(bl[0]) > 0):
        # print(f'index_ {index_
        if ins and bl['symbol'] in lsa:
            print(f'before tests t {t}')
        if (fs[index_] > 0) != (bl['delta'] > 0):
            if abs(fs[index_])/bl['wprice'] > BIG:
                if ins:# and bl['symbol'] in lsa:
                    print(f"non match first symbol {bl['symbol']}")
                continue
        if (bl['delta'] > 0) and (bl['flags'] & BL):
            if ins:# and bl['symbol'] in lsa:
                print(f"symbol {bl['symbol']} BL continue")
            continue
        if (bl['delta'] < 0) and (bl['flags'] & BH):
            if ins:# and bl['symbol'] in lsa:
                print(f"symbol {bl['symbol']} BH continue")
            continue
        if ((bl['flags'] & BH) == 0) and ((bl['flags'] & BL) == 0):
            if abs(bl['delta'])/bl['wprice'] <  BIG/2:
            # if bl['wprice'] > 1500:
                if ins and bl['symbol'] in lsa:
                    print(f"symbol {bl['symbol']} no BH or BL continue")
                continue
        # if fs[index_]:
        #     if t['buy_count'] < t['put_count']:
        #         if ins and bl['symbol'] in lsa:
        #             print(f"buy coount < put_count symbol {bl['symbol']}")
        #         continue
        # else:
        #     if ins and bl['symbol'] in lsa:
        #         print(f"put count < buy count symbol {bl['symbol']}")
        #     if t['put_count'] < t['buy_count']:
        #         continue
        # print(f't[0]  {t[0]}')
        # if ins:#  and bl['symbol'] in lsa:
        #     print(f"{t['b']['symbol']}  delta {bl['delta']}  mhMap {mhMap[count]}   count {count}  {t['buy_count']} {t['put_count']}")
        # t['b'] = bl
        # if ins and bl['symbol'] in lsa:
        #     print(f"passed tests,appending t {t}")
        pfb.append({'b':t['b'],'count': t['count']})
        pm = (1000 if dvp and not dbc else (1000 if not dvp and dbc else (100000 if dvp and dbc else 1)) )
        # pm = 100
if ins:
    print(f"len(pfb) {len(pfb)}")
    for symbol in lsa:
        d = list(filter(lambda t: t['b']['symbol']==symbol, pfb))
        if len(d) == 1:
            d = d[0]
            b = d['b']
            count = d['count']
            print(f"final number for {symbol} is {(abs(b['delta'])/(count if dbc else 1))*le*pm/(b['wprice'] if dvp else 1) - 4 * b['wprice']}")
        else:
            print(f'symbol {symbol} not in pfb')
pq = PriorityQueue()
for symbol in SYMBOLS:
    d = list(filter(lambda t: t['b']['symbol']==symbol, pfb))
    if len(d) == 1:
        d = d[0]
        b = d['b']
        count = d['count']
        v = -((abs(b['delta'])/(count if dbc else 1))*le*pm/(b['wprice'] if dvp else 1) - 4 * b['wprice'])
        pq.put((v,d))
# for l in range(len(priority_queue)):
#     print(f'{heapq.heappop(priority_queue)} {l}')

print(f"pq.qsize() {pq.qsize()}")
# pfb.sort(reverse=True, key=lambda t: (abs(t['b']['delta'])/(t['count'] if dbc else 1))*le*pm/(t['b']['wprice'] if dvp else 1) - 4 * t['b']['wprice'])
fb = []
m = (money / 2)*(wm if webull else 1)
i = 0
while not pq.empty():
# for i,t in enumerate(pfb):
    priority, t = pq.get()
    print(f"{priority:.6f} {t['b']} {i}")
    if i == 15:
        break
    i += 1
    if abs(t['b']['delta'])*le - 4 * t['b']['wprice'] < 0 and pf:
        if MSE or dvp:
            # if ins and t['b']['symbol'] in lsa:
            if ins:
                print(f"failed < 0 test {t['b']['symbol']}  t{t}")
            continue
        else:
            break

    c = round(m/t['b']['wprice'])
    # c = min(c,math.floor(money/2500))
    # x = (10 if i<11 else 5)
    x = 14
    c = min(c,x)
    # price = t['b']['wprice']
    # if price < 1000:
    #     c = min(c,5)
    # elif price < 3000:
    #     c = min(c,3)
    # elif price < 6000:
    #     c = min(c,2)
    # else:
    #     c = min(c,1)
    # c = min(c,2)
    if c > 0:
        y = t['b']
        # fb.append((y['delta'],y['wprice'],  y['aDelta'],c,y['symbol'], y['wprice'] / (wm if webull else 1)))
        fb.append({'delta': float(y['delta']),'wprice': y['wprice'],'aDelta':  y['aDelta'],'count': c,'symbol': y['symbol'],
                   'price': y['wprice'] / (wm if webull else 1)})
        m -= y['wprice'] * c



print('\n\n\n\r')
print(f'*******************money {money:.2f} *********************')
print('\n\n\n\r')
correct=[]
wrong=[]
print(f"*****************************************************Before if pf   len(fb) {len(fb)}*******************************************")
if pf:
    for t in fb:
        print(f'{t},')
else:
    moneyUsed = 0
    for t in fb:
        moneyUsed += t['wprice']*t['count']
        if np.sign(t['delta']) == np.sign(t['aDelta']):
            money += (abs(t['aDelta'])*le-t['wprice'])*t['count']
            print(f'direction correct {"call" if np.sign(t['delta']) > 0 else "put"} for {t['count']} instances \
                {(abs(t['aDelta'])*le-t['wprice'])*t['count']:.2f} symbol {t['symbol']}  {get_company_name(t['symbol'])}')
            correct.append(t['symbol'])
        else:
            money -= t['wprice']*t['count']
            print(f'direction wrong {"call" if np.sign(t['delta']) > 0 else "put"} for {t['count']} instances \
                    lost {t['wprice']*t['count']:.2f}  symbol {t['symbol']} {get_company_name(t['symbol'])}')
            wrong.append(t['symbol'])
        print(f'bet {t['wprice']*t['count']:.2f}  money {money:.2f}')
    print(f'money used was ${moneyUsed:.2f}')
    print(correct)
    print(wrong)
    print(len(correct))
    print(len(wrong))





print('\r\n\r\n')
print(f'money ${money:.2f}')
first_run_parser = True

bl {'delta': np.float64(-23.193351439229865), 'wprice': 251.42599487304685, 'aDelta': -2.9600067138671875, 'symbol': 'NVDA', 'mse': np.float64(0.0689311944756298), 'mii': 0, 'flags': 0} before
bl {'delta': np.float64(-33.13335919889981), 'wprice': 251.42599487304685, 'aDelta': -2.9600067138671875, 'symbol': 'NVDA', 'mse': np.float64(0.0689311944756298), 'mii': 0, 'flags': 0} after
before tests t {'count': 5, 'buy': np.False_, 'b': {'delta': np.float64(-33.13335919889981), 'wprice': 251.42599487304685, 'aDelta': -2.9600067138671875, 'symbol': 'NVDA', 'mse': np.float64(0.0689311944756298), 'mii': 0, 'flags': 0}, 'buy_count': 4, 'put_count': 1}
symbol AVGO BH continue
symbol ORCL BL continue
symbol DIS BL continue
symbol VZ BL continue
bl {'delta': np.float64(0.4032664258028048), 'wprice': 7066.093945312499, 'aDelta': 171.83984375, 'symbol': 'BKNG', 'mse': np.float64(0.5976907706456237), 'mii': 0, 'flags': 0} before
bl {'delta': np.float64(0.5760948940040069), 'wprice': 7066.093945312499,